In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Transpilasi dengan pengurus laluan

<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan penggunaan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</details>
Cara yang disyorkan untuk mentranspilasi Circuit ialah dengan mencipta pengurus laluan berperingkat dan kemudian melaksanakan kaedah `run`-nya dengan Circuit sebagai input. Halaman ini menerangkan cara mentranspilasi Circuit kuantum menggunakan pendekatan ini.
## Apa itu pengurus laluan (berperingkat)?
Dalam konteks Qiskit SDK, transpilasi merujuk kepada proses mengubah Circuit input kepada bentuk yang sesuai untuk dilaksanakan pada peranti kuantum. Transpilasi biasanya berlaku dalam urutan langkah yang dipanggil laluan Transpiler. Circuit diproses oleh setiap laluan Transpiler secara berurutan, dengan output satu laluan menjadi input kepada yang berikutnya. Sebagai contoh, satu laluan boleh melalui Circuit dan menggabungkan semua jujukan berturut-turut Gate satu-Qubit, dan kemudian laluan berikutnya boleh mensintesiskan Gate-Gate tersebut ke dalam set asas peranti sasaran. Laluan Transpiler yang disertakan dengan Qiskit terletak dalam modul [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes).

Pengurus laluan ialah objek yang menyimpan senarai laluan Transpiler dan boleh melaksanakannya pada Circuit. Cipta pengurus laluan dengan memulakan [`PassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) dengan senarai laluan Transpiler. Untuk menjalankan transpilasi pada Circuit, panggil kaedah [`run`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) dengan Circuit sebagai input.

Pengurus laluan berperingkat ialah sejenis pengurus laluan khas yang mewakili tahap abstraksi di atas pengurus laluan biasa. Manakala pengurus laluan biasa terdiri daripada beberapa laluan Transpiler, pengurus laluan berperingkat terdiri daripada beberapa *pengurus laluan*. Ini adalah abstraksi yang berguna kerana transpilasi biasanya berlaku dalam peringkat diskret, seperti yang diterangkan dalam [Peringkat Transpiler](transpiler-stages), di mana setiap peringkat diwakili oleh pengurus laluan. Pengurus laluan berperingkat diwakili oleh kelas [`StagedPassManager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.StagedPassManager). Selebihnya halaman ini menerangkan cara mencipta dan menyesuaikan pengurus laluan (berperingkat).
## Jana pengurus laluan berperingkat pratetap
Untuk mencipta pengurus laluan berperingkat pratetap dengan nilai lalai yang munasabah, gunakan fungsi [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager):

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

Untuk mentranspilasi Circuit atau senarai Circuit dengan pengurus laluan, hantar Circuit atau senarai Circuit ke kaedah `run`. Mari kita lakukan ini pada Circuit dua-Qubit yang terdiri daripada Hadamard diikuti dua Gate CX bersebelahan:

In [2]:
from qiskit import QuantumRegister, QuantumCircuit

# Create a circuit
qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)
a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

# Transpile it by calling the run method of the pass manager
transpiled = pass_manager.run(circuit)

# Draw it, excluding idle qubits from the diagram
transpiled.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/c1426e6c-f506-4938-8c0a-05198bae9746-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/dcc69b72-e13b-4df6-a51f-a5ef2108bae7-0.svg)

Lihat [Tetapan lalai dan pilihan konfigurasi transpilasi](defaults-and-configuration-options) untuk penerangan argumen yang mungkin bagi fungsi `generate_preset_pass_manager`. Argumen kepada `generate_preset_pass_manager` sepadan dengan argumen kepada fungsi [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<CodeAssistantAdmonition tagLine="Having trouble remembering pass manager details? Try asking Qiskit Code Assistant." />

Jika pengurus laluan pratetap tidak memenuhi keperluan anda, sesuaikan transpilasi dengan mencipta pengurus laluan (berperingkat) atau bahkan laluan transpilasi. Selebihnya halaman ini menerangkan cara mencipta pengurus laluan. Untuk arahan cara mencipta laluan transpilasi, lihat [Tulis laluan Transpiler anda sendiri](custom-transpiler-pass).

## Cipta pengurus laluan anda sendiri

Modul [qiskit.transpiler.passes](https://docs.quantum.ibm.com/api/qiskit/transpiler_passes) menyertakan banyak laluan Transpiler yang boleh digunakan untuk mencipta pengurus laluan. Untuk mencipta pengurus laluan, mulakan `PassManager` dengan senarai laluan. Sebagai contoh, kod berikut mencipta laluan Transpiler yang menggabungkan Gate dua-Qubit bersebelahan dan kemudian mensintesiskannya ke dalam asas [$R_y$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RYGate), [$R_z$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RZGate), dan [$R_{xx}$](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.library.RXXGate).

In [3]:
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import (
    Collect2qBlocks,
    ConsolidateBlocks,
    UnitarySynthesis,
)

basis_gates = ["rx", "ry", "rxx"]
translate = PassManager(
    [
        Collect2qBlocks(),
        ConsolidateBlocks(basis_gates=basis_gates),
        UnitarySynthesis(basis_gates),
    ]
)

Untuk menunjukkan pengurus laluan ini beraksi, ujilah pada Circuit dua-Qubit yang terdiri daripada Hadamard diikuti dua Gate CX bersebelahan:

In [4]:
from qiskit import QuantumRegister, QuantumCircuit

qubits = QuantumRegister(2, name="q")
circuit = QuantumCircuit(qubits)

a, b = qubits
circuit.h(a)
circuit.cx(a, b)
circuit.cx(b, a)

circuit.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/01317c54-68b5-4e41-893f-82ee223e22f0-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/bc208935-e63c-461b-90d0-a6f4cabc16b6-0.svg)

Untuk menjalankan pengurus laluan pada Circuit, panggil kaedah `run`.

In [5]:
translated = translate.run(circuit)
translated.draw("mpl")

<Image src="../docs/images/guides/transpile-with-pass-managers/extracted-outputs/019ad99b-bd38-4217-90ee-da43959dc8ad-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/transpile-with-pass-managers/extracted-outputs/adb5c242-5cba-4878-a00d-4ec47737d029-0.svg)

Untuk contoh yang lebih lanjut yang menunjukkan cara mencipta pengurus laluan untuk melaksanakan teknik penindasan ralat yang dikenali sebagai penggabungan dinamik, lihat [Cipta pengurus laluan untuk penggabungan dinamik](dynamical-decoupling-pass-manager).

## Cipta pengurus laluan berperingkat

`StagedPassManager` ialah pengurus laluan yang terdiri daripada peringkat individu, di mana setiap peringkat ditakrifkan oleh contoh `PassManager`. Anda boleh mencipta `StagedPassManager` dengan menentukan peringkat yang dikehendaki. Sebagai contoh, kod berikut mencipta pengurus laluan berperingkat dengan dua peringkat, `init` dan `translation`. Peringkat `translation` ditakrifkan oleh pengurus laluan yang dicipta sebelum ini.

In [6]:
from qiskit.transpiler import PassManager, StagedPassManager
from qiskit.transpiler.passes import UnitarySynthesis, Unroll3qOrMore

basis_gates = ["rx", "ry", "rxx"]
init = PassManager(
    [UnitarySynthesis(basis_gates, min_qubits=3), Unroll3qOrMore()]
)
staged_pm = StagedPassManager(
    stages=["init", "translation"], init=init, translation=translate
)

Tiada had pada bilangan peringkat yang boleh anda masukkan dalam pengurus laluan berperingkat.

Cara berguna lain untuk mencipta pengurus laluan berperingkat ialah dengan bermula daripada pengurus laluan berperingkat pratetap dan kemudian menukar beberapa peringkat. Sebagai contoh, kod berikut menjana pengurus laluan pratetap dengan tahap pengoptimuman 3, dan kemudian menentukan peringkat `pre_layout` tersuai.

In [7]:
import numpy as np
from qiskit.circuit.library import HGate, PhaseGate, RXGate, TdgGate, TGate
from qiskit.transpiler.passes import InverseCancellation

pass_manager = generate_preset_pass_manager(3, backend)
inverse_gate_list = [
    HGate(),
    (RXGate(np.pi / 4), RXGate(-np.pi / 4)),
    (PhaseGate(np.pi / 4), PhaseGate(-np.pi / 4)),
    (TGate(), TdgGate()),
]
logical_opt = PassManager(
    [
        InverseCancellation(inverse_gate_list),
    ]
)

# Add pre-layout stage to run extra logical optimization
pass_manager.pre_layout = logical_opt

[Fungsi penjana peringkat](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#stage-generator-functions) mungkin berguna untuk membina pengurus laluan tersuai.
Fungsi-fungsi tersebut menjana peringkat yang menyediakan fungsi umum yang digunakan dalam banyak pengurus laluan.
Sebagai contoh, [`generate_embed_passmanager`](https://docs.quantum.ibm.com/api/qiskit/transpiler_preset#qiskit.transpiler.preset_passmanagers.generate_embed_passmanager) boleh digunakan untuk menjana peringkat
bagi "membenamkan" `Layout` awal yang dipilih daripada laluan susun atur ke peranti sasaran yang ditentukan.

## Langkah seterusnya
> **Tip:** - [Tulis laluan Transpiler tersuai](custom-transpiler-pass).
>     - [Cipta pengurus laluan untuk penggabungan dinamik](dynamical-decoupling-pass-manager).
>     - Untuk mengetahui lebih lanjut tentang fungsi `generate_preset_passmanager`, lihat topik [Tetapan lalai dan pilihan konfigurasi transpilasi](defaults-and-configuration-options).
>     - Cuba panduan [Bandingkan tetapan Transpiler](/guides/circuit-transpilation-settings).
>     - Semak [dokumentasi API Transpiler.](https://docs.quantum.ibm.com/api/qiskit/transpiler)